In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/stock_data.parquet').head(1000)
## use below to run on all dataset
## df = pd.read_parquet('../data/stock_data.parquet')
print(df.columns.tolist())

['permno', 'date', 'ticker', 'company_name', 'exchange', 'open', 'high', 'low', 'close', 'volume', 'ret', 'shrout', 'mkt_cap', 'vw_mkt_ret', 'sp500_ret', 'sma_5', 'ema_5', 'sma_10', 'ema_10', 'sma_20', 'ema_20', 'sma_50', 'ema_50', 'sma_100', 'ema_100', 'sma_200', 'ema_200', 'rstd_5', 'rstd_10', 'rstd_20', 'rstd_50', 'rstd_100', 'sales_per_share', 'operating_margin', 'net_profit_margin', 'roe', 'roa', 'current_ratio', 'debt_ratio', 'mkt_cap_q', 'book_to_market']


### Define the features

In [4]:
TECHNICAL_COLS = [
    'sma_5', 'sma_10', 'sma_20', 'sma_50', 'sma_100', 'sma_200',
    'ema_5', 'ema_10', 'ema_20', 'ema_50', 'ema_100', 'ema_200',
    'rstd_5', 'rstd_10', 'rstd_20', 'rstd_50', 'rstd_100'
]

FUNDAMENTAL_COLS = [
    'sales_per_share', 'operating_margin', 'net_profit_margin',
    'roe', 'roa', 'current_ratio', 'debt_ratio', 
    'book_to_market', 'mkt_cap_q', 'mkt_cap'
]

PRICE_COLS = ['close']

FEATURE_COLS = TECHNICAL_COLS + FUNDAMENTAL_COLS + PRICE_COLS

## Checking for columns with NaNs

In [5]:
print("NaN counts per feature:")
print(df[FEATURE_COLS].isna().sum())
print(f"\nTotal rows: {len(df)}")
print(f"Rows with any NaN: {df[FEATURE_COLS].isna().any(axis=1).sum()}")

NaN counts per feature:
sma_5                  4
sma_10                 9
sma_20                19
sma_50                49
sma_100               99
sma_200              199
ema_5                  4
ema_10                 9
ema_20                19
ema_50                49
ema_100               99
ema_200              199
rstd_5                 4
rstd_10                9
rstd_20               19
rstd_50               49
rstd_100              99
sales_per_share       75
operating_margin      75
net_profit_margin     75
roe                   75
roa                   75
current_ratio         75
debt_ratio            75
book_to_market        75
mkt_cap_q             75
mkt_cap                0
close                  0
dtype: int64

Total rows: 1000
Rows with any NaN: 199


### Drop rows with nans like specified in the paper

In [6]:
df_clean = df.dropna(subset=FEATURE_COLS)

print(f"Rows before: {len(df)}")
print(f"Rows after: {len(df_clean)}")
print(f"Rows dropped: {len(df) - len(df_clean)}")

Rows before: 1000
Rows after: 801
Rows dropped: 199


### Remove the stocks that do not have enough history for the agent to learn

In [7]:
days_per_stock = df_clean.groupby('ticker').size()

# Keep only stocks with 250+ days
valid_stocks = days_per_stock[days_per_stock >= 250].index

df_clean = df_clean[df_clean['ticker'].isin(valid_stocks)]

print(f"Stocks before: {len(days_per_stock)}")
print(f"Stocks after: {len(valid_stocks)}")
print(f"Rows remaining: {len(df_clean)}")

Stocks before: 1
Stocks after: 1
Rows remaining: 801


### Split into training, validation, and testing data

In [8]:
TRAIN_END = '2020-06-01'
VAL_END = '2020-09-01'

df_train = df_clean[df_clean['date'] < TRAIN_END]
df_val = df_clean[(df_clean['date'] >= TRAIN_END) & (df_clean['date'] < VAL_END)]
df_test = df_clean[df_clean['date'] >= VAL_END]

print(f"Train: {len(df_train)} rows")
print(f"Validation: {len(df_val)} rows")
print(f"Test: {len(df_test)} rows")

Train: 801 rows
Validation: 0 rows
Test: 0 rows


### Z-score

In [9]:
train_mean = df_train[FEATURE_COLS].mean()
train_std = df_train[FEATURE_COLS].std()

# Normalize all splits using training stats
df_train[FEATURE_COLS] = (df_train[FEATURE_COLS] - train_mean) / train_std
df_val[FEATURE_COLS] = (df_val[FEATURE_COLS] - train_mean) / train_std
df_test[FEATURE_COLS] = (df_test[FEATURE_COLS] - train_mean) / train_std

print("Training data stats after normalization:")
print(df_train[FEATURE_COLS].describe().loc[['mean', 'std']])

Training data stats after normalization:
             sma_5        sma_10        sma_20        sma_50       sma_100  \
mean  3.548278e-17 -1.330604e-17  8.870696e-18  1.774139e-17  2.483795e-16   
std   1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   

           sma_200         ema_5        ema_10  ema_20        ema_50  ...  \
mean  7.096557e-17  4.435348e-18  4.878883e-17     0.0  1.774139e-17  ...   
std   1.000000e+00  1.000000e+00  1.000000e+00     1.0  1.000000e+00  ...   

      operating_margin  net_profit_margin           roe           roa  \
mean     -4.080520e-16      -8.870696e-17  2.306381e-16  7.983626e-17   
std       1.000000e+00       1.000000e+00  1.000000e+00  1.000000e+00   

      current_ratio    debt_ratio  book_to_market     mkt_cap_q       mkt_cap  \
mean   2.093484e-15 -2.430571e-15   -4.257934e-16  9.935179e-16 -4.967590e-16   
std    1.000000e+00  1.000000e+00    1.000000e+00  1.000000e+00  1.000000e+00   

             close  
mean  2

### Save clean data to use in gym

In [11]:
# Combine all splits with a split label
df_train['split'] = 'train'
df_val['split'] = 'val'
df_test['split'] = 'test'

df_final = pd.concat([df_train, df_val, df_test])

df_final.to_parquet('../data/stock_data_clean.parquet', index=False)

print(f"Saved {len(df_final)} rows to data/stock_data_clean.parquet")

Saved 801 rows to data/stock_data_clean.parquet
